# **Deber 4**
Martín Montero<br>
00328595

Para realizar el deber se optó por trabajar con Jupyter Notebook en local para que la integración con Wireshark sea fluida, sin tener que configurar pasos extra para capturar el tráfico de Google.
El código y las solicitudes son compatibles con Google Colab, pero la captura de paquetes se realizó localmente por facilidad de análisis.

## **1. Configuraciones**

### **Entorno para Descifrado TLS**
Para que Wireshark pueda "ver" dentro del tráfico HTTPS, se configura Python para que guarde las llaves de cifrado en un archivo local.

In [1]:
import os
import requests

os.environ['SSLKEYLOGFILE'] = 'sslkeys.log'

print(f"Archivo de llaves configurado en: {os.path.abspath('sslkeys.log')}")

BASE_URL = "https://jsonplaceholder.typicode.com"

Archivo de llaves configurado en: /home/martin/Desktop/Main/USFQ/Semestre_8/Redes/Teoria/Deber 4/sslkeys.log


Se agrega el archivo con las llaves de cifrado a Wireshark, en Edit > Preferences > Protocols > TLS.

<img src="https://raw.githubusercontent.com/martin2000002/Redes-Deber-4/refs/heads/main/assets/tls_wireshark.png" width="60%">

### **Headers para Filtrado**
Se crea una etiqueta personalizada para filtrar fácilmente en Wireshark

In [2]:
HEADERS_BASE = {
    'X-Redes': 'Deber-4-USFQ', # Se puede filtrar con "http contains "Deber-4-USFQ"
    'Content-Type': 'application/json'
}

## **2. Desarrollo de los Enunciados**

### **a. Recuperar todos los usuarios**
Se realiza una petición `GET` al endpoint `/users`. Se itera sobre la lista para extraer el `id` y el `name`.

In [3]:
def get_all_users():
    print("--- Lista de Usuarios ---")
    response = requests.get(f"{BASE_URL}/users", headers=HEADERS_BASE)
    
    if response.status_code == 200:
        users = response.json()
        for user in users:
            print(f"ID: {user['id']} | Nombre: {user['name']}")
    else:
        print(f"Error en la consulta: {response.status_code}")

get_all_users()

--- Lista de Usuarios ---
ID: 1 | Nombre: Leanne Graham
ID: 2 | Nombre: Ervin Howell
ID: 3 | Nombre: Clementine Bauch
ID: 4 | Nombre: Patricia Lebsack
ID: 5 | Nombre: Chelsey Dietrich
ID: 6 | Nombre: Mrs. Dennis Schulist
ID: 7 | Nombre: Kurtis Weissnat
ID: 8 | Nombre: Nicholas Runolfsdottir V
ID: 9 | Nombre: Glenna Reichert
ID: 10 | Nombre: Clementina DuBuque


#### **Análisis en Wireshark**

##### **Request:**
<img src="https://raw.githubusercontent.com/martin2000002/Redes-Deber-4/refs/heads/main/assets/a_request.png" width="80%">

##### **Response:**
<img src="https://raw.githubusercontent.com/martin2000002/Redes-Deber-4/refs/heads/main/assets/a_response.png" width="80%">

##### **Análisis de Capas:**

Tomando como referencia al cliente:
| Capa OSI | Campo | Valor Identificado |
| :--- | :--- | :--- |
| **Capa 2: Enlace** | **MAC Origen / Destino** | `30:52:cb:ea:46:b3` (PC Cliente) / `70:8c:b6:3f:a6:89` (Router) |
| **Capa 3: Red** | **IP Origen / Destino** | `2800:bf0:118:b5a:d8dd:52ef:1f38:2d0f` (IPv6 Cliente) / `2606:4700:3030::6815:3b13` (IP Servidor) |
| **Capa 4: Transporte** | **Puerto Origen / Destino** | `35784` (Efímero) / `443` (HTTPS) |

##### **Capa 7: Aplicación (Tráfico Descifrado)**

**Petición del Cliente (Request Header):**
```http
    GET /users HTTP/1.1
    Host: jsonplaceholder.typicode.com
    User-Agent: python-requests/2.33.1
    Accept-Encoding: gzip, deflate, zstd
    Accept: */*
    Connection: keep-alive
    X-Redes: Deber-4-USFQ
    Content-Type: application/json
```
> Se identifica un `GET` sin cuerpo; la cabecera `X-Redes` permite localizar fácilmente este flujo en Wireshark.

**Respuesta del Servidor (Response Header):**
```http
    HTTP/1.1 200 OK
    Date: Wed, 22 Apr 2026 04:19:18 GMT
    Content-Type: application/json; charset=utf-8
    Content-Length: 1847
    Connection: keep-alive
    access-control-allow-credentials: true
    Cache-Control: max-age=43200
    Content-Encoding: gzip
    etag: W/"160d-1eMSsxeJRfnVLRBmYJSbCiJZ1qQ"
    expires: -1
    nel: {"report_to":"heroku-nel","response_headers":["Via"],"max_age":3600,"success_fraction":0.01,"failure_fraction":0.1}
    pragma: no-cache
    report-to: {"group":"heroku-nel","endpoints":[{"url":"https://nel.heroku.com/reports?s=DLNAOqUhMrbAscoEX6HKrEYuLRPBENjfSCAgKnw0Hrw%3D\u0026sid=e11707d5-02a7-43ef-b45e-2cf4d2036f7d\u0026ts=1773308320"}],"max_age":3600}
    reporting-endpoints: heroku-nel="https://nel.heroku.com/reports?s=DLNAOqUhMrbAscoEX6HKrEYuLRPBENjfSCAgKnw0Hrw%3D&sid=e11707d5-02a7-43ef-b45e-2cf4d2036f7d&ts=1773308320"
    Server: cloudflare
    vary: Origin, Accept-Encoding
    via: 2.0 heroku-router
    x-content-type-options: nosniff
    x-powered-by: Express
    x-ratelimit-limit: 1000
    x-ratelimit-remaining: 998
    x-ratelimit-reset: 1773308335
    Age: 1169
    Accept-Ranges: bytes
    cf-cache-status: HIT
    CF-RAY: 9f01d459197ca9db-ATL
    alt-svc: h3=":443"; ma=86400
```
> El `200 OK` confirma éxito en la consulta.
**Cuerpo de la Respuesta (Payload):**

<img src="https://raw.githubusercontent.com/martin2000002/Redes-Deber-4/refs/heads/main/assets/a_response_payload.png" width="80%">

> En el payload descifrado se observa el arreglo JSON de usuarios con campos como `id` y `name`.

### **b. Recuperar posts de un usuario particular**
Se utiliza un parámetro de consulta (`params`) para filtrar los posts por `userId`.

In [4]:
def get_posts_by_user(user_id):
    print(f"\n--- Posts de Usuario {user_id} ---")
    payload = {'userId': user_id}
    response = requests.get(f"{BASE_URL}/posts", params=payload, headers=HEADERS_BASE)
    
    if response.status_code == 200:
        posts = response.json()
        for post in posts:
            print(f"Post ID: {post['id']} | Título: {post['title']}")
    else:
        print(f"Error: {response.status_code}")

get_posts_by_user(1)


--- Posts de Usuario 1 ---
Post ID: 1 | Título: sunt aut facere repellat provident occaecati excepturi optio reprehenderit
Post ID: 2 | Título: qui est esse
Post ID: 3 | Título: ea molestias quasi exercitationem repellat qui ipsa sit aut
Post ID: 4 | Título: eum et est occaecati
Post ID: 5 | Título: nesciunt quas odio
Post ID: 6 | Título: dolorem eum magni eos aperiam quia
Post ID: 7 | Título: magnam facilis autem
Post ID: 8 | Título: dolorem dolore est ipsam
Post ID: 9 | Título: nesciunt iure omnis dolorem tempora et accusantium
Post ID: 10 | Título: optio molestias id quia eum


#### **Análisis en Wireshark**

##### **Request:**
<img src="https://raw.githubusercontent.com/martin2000002/Redes-Deber-4/refs/heads/main/assets/b_request.png" width="80%">

##### **Response:**
<img src="https://raw.githubusercontent.com/martin2000002/Redes-Deber-4/refs/heads/main/assets/b_response.png" width="80%">

##### **Análisis de Capas:**

Tomando como referencia al cliente:
| Capa OSI | Campo | Valor Identificado |
| :--- | :--- | :--- |
| **Capa 2: Enlace** | **MAC Origen / Destino** | `30:52:cb:ea:46:b3` (PC Cliente) / `70:8c:b6:3f:a6:89` (Router) |
| **Capa 3: Red** | **IP Origen / Destino** | `2800:bf0:118:b5a:d8dd:52ef:1f38:2d0f` (IPv6 Cliente) / `2606:4700:3030::6815:3b13` (IP Servidor) |
| **Capa 4: Transporte** | **Puerto Origen / Destino** | `45248` (Efímero) / `443` (HTTPS) |

> El puerto de origen cambia porque el sistema operativo asigna un puerto efímero nuevo para cada conexión TCP; además, la IP de destino puede variar por CDN/DNS y balanceo de carga.

##### **Capa 7: Aplicación (Tráfico Descifrado)**

**Petición del Cliente (Request Header):**
```http
    GET /posts?userId=1 HTTP/1.1
    Host: jsonplaceholder.typicode.com
    User-Agent: python-requests/2.33.1
    Accept-Encoding: gzip, deflate, zstd
    Accept: */*
    Connection: keep-alive
    X-Redes: Deber-4-USFQ
    Content-Type: application/json
```

> Aquí se observa el filtro `userId=1` en la URL, que corresponde al uso de `params={'userId': 1}` en el código Python.

**Respuesta del Servidor (Response Header):**
```http
    HTTP/1.1 200 OK
    Date: Wed, 22 Apr 2026 04:58:31 GMT
    Content-Type: application/json; charset=utf-8
    Content-Length: 1002
    Connection: keep-alive
    access-control-allow-credentials: true
    Cache-Control: max-age=43200
    Content-Encoding: gzip
    etag: W/"aa6-j2NSH739l9uq40OywFMn7Y0C/iY"
    expires: -1
    nel: {"report_to":"heroku-nel","response_headers":["Via"],"max_age":3600,"success_fraction":0.01,"failure_fraction":0.1}
    pragma: no-cache
    report-to: {"group":"heroku-nel","endpoints":[{"url":"https://nel.heroku.com/reports?s=8i%2FId1g%2Fhu78sWavABWZA1OoCvVStDGj5qVEgBAtG9g%3D\u0026sid=e11707d5-02a7-43ef-b45e-2cf4d2036f7d\u0026ts=1774888683"}],"max_age":3600}
    reporting-endpoints: heroku-nel="https://nel.heroku.com/reports?s=8i%2FId1g%2Fhu78sWavABWZA1OoCvVStDGj5qVEgBAtG9g%3D&sid=e11707d5-02a7-43ef-b45e-2cf4d2036f7d&ts=1774888683"
    Server: cloudflare
    vary: Origin, Accept-Encoding
    via: 2.0 heroku-router
    x-content-type-options: nosniff
    x-powered-by: Express
    x-ratelimit-limit: 1000
    x-ratelimit-remaining: 999
    x-ratelimit-reset: 1774888733
    Age: 12402
    Accept-Ranges: bytes
    cf-cache-status: HIT
    CF-RAY: 9f020dcb0ddbe155-ATL
    alt-svc: h3=":443"; ma=86400
```

> El `200 OK` confirma que el filtro fue procesado correctamente y que la API devolvió datos JSON del usuario solicitado.

**Cuerpo de la Respuesta (Payload):**
<img src="https://raw.githubusercontent.com/martin2000002/Redes-Deber-4/refs/heads/main/assets/b_response_payload.png" width="80%">

> En el payload se listan únicamente los posts asociados al `userId=1`, validando el criterio de búsqueda.

### **c. Crear un nuevo post**
Se utiliza el método `POST`. Enviamos un objeto JSON en el cuerpo de la petición.

In [5]:
def create_post():
    print("\n--- Nuevo Post ---")
    new_post = {
        'title': 'Redes USFQ',
        'body': 'Analizando tráfico con Wireshark',
        'userId': 1
    }
    
    response = requests.post(f"{BASE_URL}/posts", json=new_post, headers=HEADERS_BASE)
    
    if response.status_code == 201:
        print("Post creado exitosamente (Simulado).")
        print(f"Respuesta del servidor: {response.json()}")
    else:
        print(f"Error: {response.status_code}")

create_post()


--- Nuevo Post ---
Post creado exitosamente (Simulado).
Respuesta del servidor: {'title': 'Redes USFQ', 'body': 'Analizando tráfico con Wireshark', 'userId': 1, 'id': 101}


#### **Análisis en Wireshark**

##### **Request:**
<img src="https://raw.githubusercontent.com/martin2000002/Redes-Deber-4/refs/heads/main/assets/c_request.png" width="80%">

##### **Response:**
<img src="https://raw.githubusercontent.com/martin2000002/Redes-Deber-4/refs/heads/main/assets/c_response.png" width="80%">

##### **Análisis de Capas:**

Tomando como referencia al cliente:
| Capa OSI | Campo | Valor Identificado |
| :--- | :--- | :--- |
| **Capa 2: Enlace** | **MAC Origen / Destino** | `30:52:cb:ea:46:b3` (PC Cliente) / `70:8c:b6:3f:a6:89` (Router) |
| **Capa 3: Red** | **IP Origen / Destino** | `2800:bf0:118:b5a:d8dd:52ef:1f38:2d0f` (IPv6 Cliente) / `2606:4700:3031::ac43:a797` (IP Servidor) |
| **Capa 4: Transporte** | **Puerto Origen / Destino** | `59580` (Efímero) / `443` (HTTPS) |

> En este caso cambia la IP de destino y el puerto efímero porque se establece otra conexión TCP/TLS y el servicio puede resolverse a otro nodo de la CDN.

##### **Capa 7: Aplicación (Tráfico Descifrado)**

**Petición del Cliente (Request Header):**
```http
    POST /posts HTTP/1.1
    Host: jsonplaceholder.typicode.com
    User-Agent: python-requests/2.33.1
    Accept-Encoding: gzip, deflate, zstd
    Accept: */*
    Connection: keep-alive
    X-Redes: Deber-4-USFQ
    Content-Type: application/json
    Content-Length: 85
```

> Se observa el método `POST` que crea un recurso nuevo.

**Cuerpo de la Petición (Payload):**  
<img src="https://raw.githubusercontent.com/martin2000002/Redes-Deber-4/refs/heads/main/assets/c_request_payload.png" width="80%">

> El payload contiene los campos `title`, `body` y `userId`, tal como se definieron en el diccionario `new_post`.

**Respuesta del Servidor (Response Header):**
```http
    HTTP/1.1 201 Created
    Date: Wed, 22 Apr 2026 05:02:51 GMT
    Content-Type: application/json; charset=utf-8
    Content-Length: 102
    Connection: keep-alive
    access-control-allow-credentials: true
    access-control-expose-headers: Location
    Cache-Control: no-cache
    etag: W/"66-Y7RrWqoAP9PzGe2g4FPi/evlWpE"
    expires: -1
    location: https://jsonplaceholder.typicode.com/posts/101
    nel: {"report_to":"heroku-nel","response_headers":["Via"],"max_age":3600,"success_fraction":0.01,"failure_fraction":0.1}
    pragma: no-cache
    report-to: {"group":"heroku-nel","endpoints":[{"url":"https://nel.heroku.com/reports?s=wm0lId9bp6z9txrV1ItDHppJWKwDmi5wQ1CCOqwPMSQ%3D\u0026sid=e11707d5-02a7-43ef-b45e-2cf4d2036f7d\u0026ts=1776834171"}],"max_age":3600}
    reporting-endpoints: heroku-nel="https://nel.heroku.com/reports?s=wm0lId9bp6z9txrV1ItDHppJWKwDmi5wQ1CCOqwPMSQ%3D&sid=e11707d5-02a7-43ef-b45e-2cf4d2036f7d&ts=1776834171"
    Server: cloudflare
    vary: Origin, X-HTTP-Method-Override, Accept-Encoding
    via: 2.0 heroku-router
    x-content-type-options: nosniff
    x-powered-by: Express
    x-ratelimit-limit: 1000
    x-ratelimit-remaining: 999
    x-ratelimit-reset: 1776834216
    cf-cache-status: DYNAMIC
    CF-RAY: 9f0214246a21dd21-ATL
    alt-svc: h3=":443"; ma=86400
```

> El código `201 Created` confirma que el `POST` fue aceptado y por eso se tuvo que haber simulado su creación.

**Cuerpo de la Respuesta (Payload):**
<img src="https://raw.githubusercontent.com/martin2000002/Redes-Deber-4/refs/heads/main/assets/c_response_payload.png" width="80%">

### **d. Actualizar un determinado post**
Se utiliza el método `PUT` para actualizar el recurso completo en el ID especificado.

In [6]:
def update_post(post_id):
    print(f"\n--- Actualizar Post {post_id} ---")
    updated_data = {
        'id': post_id,
        'title': 'Título Actualizado',
        'body': 'Contenido modificado para el deber 4',
        'userId': 1
    }
    
    response = requests.put(f"{BASE_URL}/posts/{post_id}", json=updated_data, headers=HEADERS_BASE)
    
    if response.status_code == 200:
        print("Post actualizado con éxito.")
        print(f"Datos recibidos: {response.json()}")
    else:
        print(f"Error: {response.status_code}")

update_post(1)


--- Actualizar Post 1 ---
Post actualizado con éxito.
Datos recibidos: {'id': 1, 'title': 'Título Actualizado', 'body': 'Contenido modificado para el deber 4', 'userId': 1}


#### **Análisis en Wireshark**

##### **Request:**
<img src="https://raw.githubusercontent.com/martin2000002/Redes-Deber-4/refs/heads/main/assets/d_request.png" width="80%">

##### **Response:**
<img src="https://raw.githubusercontent.com/martin2000002/Redes-Deber-4/refs/heads/main/assets/d_response.png" width="80%">

##### **Análisis de Capas:**

Tomando como referencia al cliente:
| Capa OSI | Campo | Valor Identificado |
| :--- | :--- | :--- |
| **Capa 2: Enlace** | **MAC Origen / Destino** | `30:52:cb:ea:46:b3` (PC Cliente) / `70:8c:b6:3f:a6:89` (Router) |
| **Capa 3: Red** | **IP Origen / Destino** | `2800:bf0:118:b5a:d8dd:52ef:1f38:2d0f` (IPv6 Cliente) / `2606:4700:3030::6815:3b13` (IP Servidor) |
| **Capa 4: Transporte** | **Puerto Origen / Destino** | `40902` (Efímero) / `443` (HTTPS) |

##### **Capa 7: Aplicación (Tráfico Descifrado)**

**Petición del Cliente (Request Header):**
```http
    PUT /posts/1 HTTP/1.1
    Host: jsonplaceholder.typicode.com
    User-Agent: python-requests/2.33.1
    Accept-Encoding: gzip, deflate, zstd
    Accept: */*
    Connection: keep-alive
    X-Redes: Deber-4-USFQ
    Content-Type: application/json
    Content-Length: 106
```

> Es un `PUT` sobre `/posts/1`; esta operación reemplaza la representación del recurso con los datos enviados en el cuerpo.

**Cuerpo de la Petición (Payload):**  
<img src="https://raw.githubusercontent.com/martin2000002/Redes-Deber-4/refs/heads/main/assets/d_request_payload.png" width="80%">

> El payload incluye `id`, `title`, `body` y `userId`, por eso su tamaño (`Content-Length`) es mayor que en operaciones sin cuerpo.

**Respuesta del Servidor (Response Header):**
```http
    HTTP/1.1 200 OK
    Date: Wed, 22 Apr 2026 05:17:31 GMT
    Content-Type: application/json; charset=utf-8
    Transfer-Encoding: chunked
    Connection: keep-alive
    access-control-allow-credentials: true
    Cache-Control: no-cache
    etag: W/"70-LJhQSmDa7+MNkDby1xMHoFfmlIE"
    expires: -1
    nel: {"report_to":"heroku-nel","response_headers":["Via"],"max_age":3600,"success_fraction":0.01,"failure_fraction":0.1}
    pragma: no-cache
    report-to: {"group":"heroku-nel","endpoints":[{"url":"https://nel.heroku.com/reports?s=sXGgjsqG4LSzCwA1nkqh4oY7UYvtBIwXtQKbemvo14Q%3D\u0026sid=e11707d5-02a7-43ef-b45e-2cf4d2036f7d\u0026ts=1776835051"}],"max_age":3600}
    reporting-endpoints: heroku-nel="https://nel.heroku.com/reports?s=sXGgjsqG4LSzCwA1nkqh4oY7UYvtBIwXtQKbemvo14Q%3D&sid=e11707d5-02a7-43ef-b45e-2cf4d2036f7d&ts=1776835051"
    Server: cloudflare
    vary: Origin, Accept-Encoding
    via: 2.0 heroku-router
    x-content-type-options: nosniff
    x-powered-by: Express
    x-ratelimit-limit: 1000
    x-ratelimit-remaining: 999
    x-ratelimit-reset: 1776835056
    cf-cache-status: DYNAMIC
    Content-Encoding: zstd
    CF-RAY: 9f02299bfe6f9afd-GYE
    alt-svc: h3=":443"; ma=86400
```

> El `200 OK` confirma la actualización simulada; además, `chunked` y `zstd` describen cómo se transmitió y comprimió la respuesta.

**Cuerpo de la Respuesta (Payload):**
<img src="https://raw.githubusercontent.com/martin2000002/Redes-Deber-4/refs/heads/main/assets/d_response_payload.png" width="80%">

### **e. Eliminar un determinado post**
Se utiliza el método `DELETE`.

In [ ]:
def delete_post(post_id):
    print(f"\n--- Eliminar Post {post_id} ---")
    response = requests.delete(f"{BASE_URL}/posts/{post_id}", headers=HEADERS_BASE)
    
    if response.status_code == 200:
        print(f"El post {post_id} ha sido eliminado correctamente.")
    else:
        print(f"Error: {response.status_code}")

delete_post(1)


--- liminar Post 1 ---
El post 1 ha sido eliminado correctamente.


#### **Análisis en Wireshark**

##### **Request:**
<img src="https://raw.githubusercontent.com/martin2000002/Redes-Deber-4/refs/heads/main/assets/e_request.png" width="80%">

##### **Response:**
<img src="https://raw.githubusercontent.com/martin2000002/Redes-Deber-4/refs/heads/main/assets/e_response.png" width="80%">

##### **Análisis de Capas:**

Tomando como referencia al cliente:
| Capa OSI | Campo | Valor Identificado |
| :--- | :--- | :--- |
| **Capa 2: Enlace** | **MAC Origen / Destino** | `30:52:cb:ea:46:b3` (PC Cliente) / `70:8c:b6:3f:a6:89` (Router) |
| **Capa 3: Red** | **IP Origen / Destino** | `2800:bf0:118:b5a:d8dd:52ef:1f38:2d0f` (IPv6 Cliente) / `2606:4700:3031::ac43:a797` (IP Servidor) |
| **Capa 4: Transporte** | **Puerto Origen / Destino** | `41522` (Efímero) / `443` (HTTPS) |

##### **Capa 7: Aplicación (Tráfico Descifrado)**

**Petición del Cliente (Request Header):**
```http
    DELETE /posts/1 HTTP/1.1
    Host: jsonplaceholder.typicode.com
    User-Agent: python-requests/2.33.1
    Accept-Encoding: gzip, deflate, zstd
    Accept: */*
    Connection: keep-alive
    X-Redes: Deber-4-USFQ
    Content-Type: application/json
    Content-Length: 0
```

> En `DELETE` no se envía cuerpo (`Content-Length: 0`); la operación se define por la URI del recurso (`/posts/1`).

**Respuesta del Servidor (Response Header):**
```http
    HTTP/1.1 200 OK
    Date: Wed, 22 Apr 2026 05:25:04 GMT
    Content-Type: application/json; charset=utf-8
    Content-Length: 2
    Connection: keep-alive
    access-control-allow-credentials: true
    Cache-Control: no-cache
    etag: W/"2-vyGp6PvFo4RvsFtPoIWeCReyIC8"
    expires: -1
    nel: {"report_to":"heroku-nel","response_headers":["Via"],"max_age":3600,"success_fraction":0.01,"failure_fraction":0.1}
    pragma: no-cache
    report-to: {"group":"heroku-nel","endpoints":[{"url":"https://nel.heroku.com/reports?s=NoQU1S4Z83L05kqkS%2Fl82%2BXW5OPAt6lf5TVfYCpP6j0%3D\u0026sid=e11707d5-02a7-43ef-b45e-2cf4d2036f7d\u0026ts=1776835503"}],"max_age":3600}
    reporting-endpoints: heroku-nel="https://nel.heroku.com/reports?s=NoQU1S4Z83L05kqkS%2Fl82%2BXW5OPAt6lf5TVfYCpP6j0%3D&sid=e11707d5-02a7-43ef-b45e-2cf4d2036f7d&ts=1776835503"
    Server: cloudflare
    vary: Origin, Accept-Encoding
    via: 2.0 heroku-router
    x-content-type-options: nosniff
    x-powered-by: Express
    x-ratelimit-limit: 1000
    x-ratelimit-remaining: 999
    x-ratelimit-reset: 1776835536
    cf-cache-status: DYNAMIC
    CF-RAY: 9f0234a95efc824d-GYE
    alt-svc: h3=":443"; ma=86400
```

> El `200 OK` confirma que la petición fue procesada y, en JSONPlaceholder, la eliminación es simulada.

**Cuerpo de la Respuesta (Payload):**
<img src="https://raw.githubusercontent.com/martin2000002/Redes-Deber-4/refs/heads/main/assets/e_response_payload.png" width="80%">

> El payload mínimo (`{}`) es esperado porque esta API fake no mantiene estado real y solo emula una eliminación exitosa.